# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

> **Solved version.** Every original prompt is kept; each TODO is filled in and a written answer follows
> each challenge. It runs from `Challenge_Solutions/` inside the project, which has the same `database/`,
> `data/` and `api/` folders as the classroom pack.

In [1]:
# Setup - needs pandas
import json
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# find the folder that holds database/flasheats.db (the project root, one level up from this notebook)
BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "database" / "flasheats.db").exists())
DB_PATH = BASE / "database" / "flasheats.db"
print("Pack root found:", BASE.name, "| database:", DB_PATH.exists())

Pack root found: sst-FDE-ass-2-flasheats-pipeline | database: True


## Load the updated FlashEats data

In [2]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13)
tickets: (202, 5)
restaurant_status: (502, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


,order_id,restaurant_id,status,last_updated_at
0,O00676,R040,READY,2026-08-17T13:29:00
1,O01506,R055,Ready,2026-08-27T16:37:00
2,O01484,R032,ready,2026-08-08T19:44:00
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00


{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the “56% late” claim?


Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order |  |  |  |
| Delivered orders have completion time |  |  |  |
| Promised ETA is valid |  |  |  |
| Event chronology is valid |  |  |  |
| “Late” has an agreed definition |  |  |  |

**Hint:** Don't start by cleaning. Ask: *what could make 56% misleading?*

In [3]:
print("Rows:", len(orders))
print("Unique orders:", orders["order_id"].nunique())
print(orders["final_status"].value_counts(dropna=False))

first = orders.drop_duplicates("order_id", keep="first").copy()
for c in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    first[c] = pd.to_datetime(first[c], format="mixed", errors="coerce")
first["status"] = first["final_status"].str.strip().str.lower()

# business grain: what differs between the duplicated rows?
dups = orders[orders["order_id"].duplicated(keep=False)].sort_values("order_id")
display(dups[["order_id", "final_status", "promised_eta", "actual_delivery_at", "traffic_bucket"]])

delivered = first[first["status"] == "delivered"]
contract = pd.DataFrame([
    ("One row = one business order", "order_id unique", f"{len(orders)} rows, {orders['order_id'].nunique()} orders",
     "WARN", "3 duplicates differ only on traffic_bucket - keep first"),
    ("Delivered orders have completion time", "actual_delivery_at not null when delivered",
     f"{delivered['actual_delivery_at'].isna().sum()} of {len(delivered)} missing", "WARN", "exclude and report; ask why"),
    ("Promised ETA is valid", "promised_eta >= created_at",
     f"{(first['promised_eta'] < first['created_at']).sum()} orders fail", "WARN", "exclude those orders from the KPI"),
    ("Event chronology is valid", "pickup_at <= actual_delivery_at",
     f"{(first['actual_delivery_at'] < first['pickup_at']).sum()} orders fail", "WARN", "exclude from time-split analysis"),
    ("'Late' has an agreed definition", "one documented definition with an owner",
     f"{len(metric_notes['stakeholders'])} stakeholder definitions, no owner", "FAIL", "needs VP Operations to decide"),
], columns=["business assumption", "data expectation", "result", "status", "action"])
contract

Rows: 1603
Unique orders: 1600
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64


,order_id,final_status,promised_eta,actual_delivery_at,traffic_bucket
119,O00120,delivered,2026-08-13T19:40:24,2026-08-13T19:34:08.431508,severe
1600,O00120,delivered,2026-08-13T19:40:24,2026-08-13T19:34:08.431508,medium
722,O00723,delivered,2026-08-05T22:19:36,2026-08-05T22:04:30.928322,medium
1601,O00723,delivered,2026-08-05T22:19:36,2026-08-05T22:04:30.928322,high
1301,O01302,delivered,2026-08-26T22:15:50.273095,2026-08-26T22:20:27.428561,medium
1602,O01302,delivered,2026-08-26T22:15:50.273095,2026-08-26T22:20:27.428561,high


,business assumption,data expectation,result,status,action
0,One row = one business order,order_id unique,"1603 rows, 1600 orders",WARN,3 duplicates differ only on traffic_bucket - keep first
1,Delivered orders have completion time,actual_delivery_at not null when delivered,37 of 1532 missing,WARN,exclude and report; ask why
2,Promised ETA is valid,promised_eta >= created_at,4 orders fail,WARN,exclude those orders from the KPI
3,Event chronology is valid,pickup_at <= actual_delivery_at,5 orders fail,WARN,exclude from time-split analysis
4,'Late' has an agreed definition,one documented definition with an owner,"4 stakeholder definitions, no owner",FAIL,needs VP Operations to decide


**Answer - Challenge 1: the validation contract**

| Business assumption | Data expectation | How it was tested | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` is unique | count rows vs unique IDs: 1,603 vs 1,600 | Medium - duplicates double-count; here the 3 copies differ only on `traffic_bucket`, so keeping the first is safe |
| Delivered orders have completion time | `actual_delivery_at` not null for delivered | 37 of 1,532 delivered orders have none | High - they silently drop out of the denominator |
| Promised ETA is valid | `promised_eta >= created_at` | 4 orders promise delivery before they were placed | Medium - those can't be judged late or on time |
| Event chronology is valid | `pickup_at <= actual_delivery_at` | 5 orders "delivered" before pickup | Low for the KPI (it doesn't use pickup), high for any pickup/drive analysis |
| "Late" has an agreed definition | one definition with an owner | `client_metric_definitions.json` lists 4 and says no owner is documented | **High** - the number changes with the definition (Challenge 2) |

What could make "56%" misleading: it depends on which orders are in the denominator and which definition of late is
used - and neither is agreed.

# Challenge 2 — Stakeholders disagree on “late”


Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|

Then answer:

> **Which one should leadership publish, and who must own that decision?**

In [4]:
analysis = orders.drop_duplicates("order_id", keep="first").copy()

for c in ["promised_eta", "actual_delivery_at", "created_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")

analysis["delay_min"] = (
    analysis["actual_delivery_at"] - analysis["promised_eta"]
).dt.total_seconds() / 60

print(json.dumps(metric_notes["stakeholders"], indent=2))

status = analysis["final_status"].str.strip().str.lower()
timed = analysis[(status == "delivered") & analysis["actual_delivery_at"].notna()]
valid_promise = timed[timed["promised_eta"] >= timed["created_at"]]
credited = pd.read_csv(BASE / "data" / "order_interventions.csv").query("intervention_type == 'CUSTOMER_CREDIT'")["order_id"]
no_credit = valid_promise[~valid_promise["order_id"].isin(credited)]

def rate(df, late_mask):
    return pd.Series({"late": int(late_mask.sum()), "orders": len(df), "late_rate_%": round(late_mask.mean() * 100, 1)})

definitions = pd.DataFrame({
    "Data Team history: delivered, non-null time, delay > 0": rate(timed, timed["delay_min"] > 0),
    "VP Operations: any delay > 0 (valid promises only)": rate(valid_promise, valid_promise["delay_min"] > 0),
    "Support Lead: delay > 10 minutes": rate(valid_promise, valid_promise["delay_min"] > 10),
    "Finance: also without orders that got a customer credit": rate(no_credit, no_credit["delay_min"] > 0),
}).T
definitions

{
  "VP Operations": "Any delivered order after the promised ETA is late.",
  "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
  "Finance": "Cancelled/refunded orders should not count in operational performance.",
  "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
}


,late,orders,late_rate_%
"Data Team history: delivered, non-null time, delay > 0",843.0,1495.0,56.4
VP Operations: any delay > 0 (valid promises only),839.0,1491.0,56.3
Support Lead: delay > 10 minutes,345.0,1491.0,23.1
Finance: also without orders that got a customer credit,798.0,1429.0,55.8


**Answer - Challenge 2**

| Definition | Late rate | Business meaning |
|---|---:|---|
| Data Team history (delivered, non-null time) | **56.4%** (843/1,495) | this is where the "56%" comes from |
| VP Operations: any delay > 0, valid promises only | 56.3% (839/1,491) | any broken promise counts - the customer's view |
| Support Lead: delay > 10 minutes | **23.1%** (345/1,491) | only delays customers really notice and complain about |
| Finance: without credited orders | 55.8% (798/1,429) | operational performance without orders already compensated |

The same data gives anything from **23% to 56%**. None of these is "wrong" - they answer different questions.

**Which one should leadership publish?** The *any delay against the original promise* rate (about 56%), because it
matches what the customer was promised - but published **with its definition written next to it**, and with the
over-10-minutes rate shown alongside.

**Who must own that decision?** VP Operations, as the owner of the delivery outcome - not the data team. Right now no
owner is documented, so this is not something the data can settle.

# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

Remember:

> `"Delivered"` vs `"delivered"` is likely representation.  
> `"handoff"` vs `"handed_off"` may be semantics.

In [5]:
def show_values(label, series):
    normalise = lambda s: s.astype("string").str.strip().str.lower().str.replace(" ", "_")
    table = series.value_counts(dropna=False).rename("rows").reset_index()
    table.columns = ["raw_value", "rows"]
    table["normalised"] = normalise(table["raw_value"])
    table["changed_by_normalising"] = table["raw_value"].astype("string") != table["normalised"]
    print(f"\n== {label}: {series.nunique()} raw values -> {normalise(series).nunique()} after fixing case/spaces")
    display(table)

show_values("final_status", orders["final_status"])
show_values("traffic_bucket", orders["traffic_bucket"])
show_values("restaurant status", restaurant_status["status"])
show_values("support category", tickets["category"])


== final_status: 3 raw values -> 2 after fixing case/spaces


,raw_value,rows,normalised,changed_by_normalising
0,delivered,1530,delivered,False
1,cancelled,68,cancelled,False
2,Delivered,5,delivered,True



== traffic_bucket: 5 raw values -> 4 after fixing case/spaces


,raw_value,rows,normalised,changed_by_normalising
0,medium,635,medium,False
1,high,448,high,False
2,low,381,low,False
3,severe,136,severe,False
4,HIGH,3,high,True



== restaurant status: 8 raw values -> 5 after fixing case/spaces


,raw_value,rows,normalised,changed_by_normalising
0,preparing,171,preparing,False
1,handed_off,168,handed_off,False
2,ready,157,ready,False
3,ready,2,ready,True
4,READY,1,ready,True
5,Ready,1,ready,True
6,handoff,1,handoff,False
7,unknown,1,unknown,False



== support category: 9 raw values -> 7 after fixing case/spaces


,raw_value,rows,normalised,changed_by_normalising
0,late_delivery,38,late_delivery,False
1,eta_changed,37,eta_changed,False
2,restaurant_delay,37,restaurant_delay,False
3,ready_but_waiting,33,ready_but_waiting,False
4,status_mismatch,29,status_mismatch,False
5,driver_not_moving,25,driver_not_moving,False
6,Late Delivery,1,late_delivery,True
7,late_delivery,1,late_delivery,True
8,ETA issue,1,eta_issue,True


**Answer - Challenge 3**

| Field | Observed values | Safe to normalise (representation) | Needs owner confirmation (semantics) |
|---|---|---|---|
| `final_status` | delivered, cancelled, Delivered | `Delivered` -> `delivered` | - |
| `traffic_bucket` | low, medium, high, severe, HIGH | `HIGH` -> `high` | what separates *high* from *severe*? |
| restaurant `status` | preparing, ready, handed_off, READY, Ready, "ready ", handoff, unknown | case and trailing spaces -> `ready` | **`handoff` vs `handed_off`** may be different states; **`unknown`** isn't a state at all - ask Restaurant Ops |
| support `category` | late_delivery, eta_changed, ..., Late Delivery, "late_delivery ", ETA issue | `Late Delivery`, `late_delivery ` -> `late_delivery` | **`ETA issue`** - is it `eta_changed` or `late_delivery`? ask the Support Lead |

Rule used: fix only *how a value is written*. Merging values that might *mean* different things is left to the owner.

# Challenge 4 — Cross-source integrity


Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

Produce:

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|

Then discuss:

> If 1% is unmapped, is that acceptable?

It depends on which business decision uses those records.

In [6]:
def coverage(label, keys, target, used_for):
    keys = keys.astype("string")
    mapped = keys.isin(target.astype("string"))
    return {"relationship": label, "rows": len(keys), "blank_ids": int(keys.isna().sum()),
            "coverage_%": round(mapped.mean() * 100, 2),
            "status": "PASS" if mapped.all() else "WARN", "used_for": used_for}

first_orders = orders.drop_duplicates("order_id")
mapping = pd.DataFrame([
    coverage("orders.restaurant_id -> restaurants", first_orders["restaurant_id"], restaurants["restaurant_id"], "restaurant breakdowns"),
    coverage("orders.driver_id -> drivers", first_orders["driver_id"], drivers["driver_id"], "driver breakdowns"),
    coverage("tickets.order_id -> orders", tickets["order_id"], first_orders["order_id"], "linking complaints to orders"),
    coverage("restaurant_status.order_id -> orders", restaurant_status["order_id"], first_orders["order_id"], "restaurant freshness"),
])
mapping

,relationship,rows,blank_ids,coverage_%,status,used_for
0,orders.restaurant_id -> restaurants,1600,3,99.81,WARN,restaurant breakdowns
1,orders.driver_id -> drivers,1600,3,99.81,WARN,driver breakdowns
2,tickets.order_id -> orders,202,3,98.51,WARN,linking complaints to orders
3,restaurant_status.order_id -> orders,502,0,100.00,PASS,restaurant freshness


**Answer - Challenge 4**

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|
| orders.restaurant_id -> restaurants | 99.81% | WARN | 3 orders have a blank restaurant_id - fine for the late rate, they just drop out of restaurant rankings |
| orders.driver_id -> drivers | 99.81% | WARN | 3 blank driver_ids; also the orders table keeps the *first* driver - use dispatch for attribution |
| tickets.order_id -> orders | 98.51% | WARN | 3 tickets have no order_id; every ticket that has one maps |
| restaurant_status.order_id -> orders | 100% | PASS | all map - but only a third of orders have a status at all |

**Is 1% unmapped acceptable?** For the company-wide late rate, yes - those orders don't need a restaurant or driver
to be counted. For a decision about a **specific restaurant or driver** (for example penalising them), no - every
unmapped order is a record that can't be attributed, and it could be the one that matters.

# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

In [7]:
rs = restaurant_status.copy()
rs["status_clean"] = rs["status"].str.strip().str.lower()
rs["last_updated_at"] = pd.to_datetime(rs["last_updated_at"], format="mixed", errors="coerce")

m = rs.merge(first[["order_id", "created_at", "pickup_at", "actual_delivery_at"]], on="order_id", how="left")
m["min_after_created"] = (m["last_updated_at"] - m["created_at"]).dt.total_seconds() / 60
m["min_vs_pickup"] = (m["last_updated_at"] - m["pickup_at"]).dt.total_seconds() / 60

print(f"Orders with any status record: {rs['order_id'].nunique()} of {first['order_id'].nunique()} "
      f"({rs['order_id'].nunique() / first['order_id'].nunique():.0%})")
print("Orders with more than one status row:", int(rs["order_id"].duplicated().sum()), "(only the latest status is kept - no history)")
print("Updates that happened AFTER pickup:", int((m["min_vs_pickup"] > 0).sum()), "of", len(m))
display(m.groupby("status_clean").agg(rows=("order_id", "size"),
                                      median_min_after_order=("min_after_created", "median"),
                                      median_min_vs_pickup=("min_vs_pickup", "median"),
                                      share_updated_after_pickup=("min_vs_pickup", lambda s: (s > 0).mean())).round(2))

Orders with any status record: 500 of 1600 (31%)
Orders with more than one status row: 2 (only the latest status is kept - no history)
Updates that happened AFTER pickup: 166 of 502


,rows,median_min_after_order,median_min_vs_pickup,share_updated_after_pickup
status_clean,,,,
handed_off,168,19.0,-6.41,0.33
handoff,1,22.0,8.10,1.00
preparing,171,19.0,-7.33,0.29
ready,161,24.0,-5.41,0.37
unknown,1,17.0,-3.22,0.00


**Answer - Challenge 5**

Restaurant status covers only **500 of 1,600 orders (31%)**, keeps only the **latest** status (no history), and **166
of 502 updates land after the driver already picked up** - even some "ready" updates arrive after the food left.

| Use case | Fresh enough? | Why |
|---|---|---|
| Weekly analytics | **WARN** | usable as a rough signal of restaurant behaviour, if the 31% coverage is stated |
| Live customer ETA | **FAIL** | updates arrive late or not at all - showing them live would mislead customers |
| Restaurant accountability | **FAIL** | without a history and a reliable "ready" time, we can't prove when food was ready - not fair to penalise anyone on this |

Same records, three different answers - freshness depends on the decision.

# Challenge 6 — Build the validation gate


Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain |  |  |  |
| Timestamp chronology |  |  |  |
| KPI definition |  |  |  |
| Category semantics |  |  |  |
| Cross-source mapping |  |  |  |
| Freshness |  |  |  |

Use only:
**PASS / WARN / FAIL / UNKNOWN**

Finally answer:

> **Should leadership publish “Late Delivery Rate = 56%” today?**

What exactly must happen before the metric becomes publishable?

In [8]:
validation_report = {
    "business_grain":       ("WARN", "1,603 rows / 1,600 orders; copies differ only on traffic_bucket", "keep first copy, log it"),
    "timestamp_chronology": ("WARN", "37 delivered without time, 4 ETA before order, 5 delivered before pickup", "exclude and report"),
    "kpi_definition":       ("FAIL", "4 stakeholder definitions give 23%-56%; no owner", "VP Operations to choose and sign off"),
    "category_semantics":   ("WARN", "case/space variants fixed; 'handoff' and 'ETA issue' unresolved", "owners to confirm"),
    "cross_source_mapping": ("WARN", "99.8% restaurants/drivers, 98.5% tickets map", "fine for the KPI, not for blame"),
    "freshness":            ("FAIL", "restaurant status: 31% coverage, no history, 166 updates after pickup", "not for live ETA or accountability"),
    "publish_56_percent":   ("NO - not as a bare '56%'", "the number is right under one definition only", "publish with the definition once the owner confirms it"),
}
pd.DataFrame(validation_report, index=["status", "evidence", "action"]).T

,status,evidence,action
business_grain,WARN,"1,603 rows / 1,600 orders; copies differ only on traffic_bucket","keep first copy, log it"
timestamp_chronology,WARN,"37 delivered without time, 4 ETA before order, 5 delivered before pickup",exclude and report
kpi_definition,FAIL,4 stakeholder definitions give 23%-56%; no owner,VP Operations to choose and sign off
category_semantics,WARN,case/space variants fixed; 'handoff' and 'ETA issue' unresolved,owners to confirm
cross_source_mapping,WARN,"99.8% restaurants/drivers, 98.5% tickets map","fine for the KPI, not for blame"
freshness,FAIL,"restaurant status: 31% coverage, no history, 166 updates after pickup",not for live ETA or accountability
publish_56_percent,NO - not as a bare '56%',the number is right under one definition only,publish with the definition once the owner confirms it


**Answer - Challenge 6: should leadership publish "Late Delivery Rate = 56%" today? Not as it stands.**

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain | WARN | 1,603 rows for 1,600 orders; copies differ only on traffic | keep the first copy and log it |
| Timestamp chronology | WARN | 37 delivered with no time, 4 promises before the order, 5 delivered before pickup | exclude from the KPI and report the counts |
| KPI definition | **FAIL** | 4 definitions give 23% to 56%; no documented owner | VP Operations picks one and signs off |
| Category semantics | WARN | case/spaces fixed; `handoff` and `ETA issue` unclear | owners confirm before merging |
| Cross-source mapping | WARN | 99.8% / 98.5% coverage | fine for the company KPI, not for blaming restaurants or drivers |
| Freshness | FAIL | restaurant status: 31% coverage, no history, often updated after pickup | not used for live ETA or accountability |

**What must happen before it's publishable:**

1. VP Operations confirms the definition (late = after the *original* promise, any minute) and owns the KPI.
2. The number is published *with* that definition and its population: "56.4% of 1,495 delivered orders with a delivery time".
3. The 37 untimed deliveries and 4 invalid promises are explained (the driver app can fill the 37 - see Class 5).
4. Nobody uses restaurant status to judge restaurants until it's reliable.

# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**